# Step 2 Summarize the scrapped data
## Goal
- Generate short, high-quality descriptions that improve RAG retrieval and LLM performance.

## Pipeline

1. Navigate to the Summary folder.
2. Upload Bielik model for Onyxia workspace.  Download file `Bielik-1.5B-v3.0-Instruct.Q8_0.gguf` from [here](https://huggingface.co/speakleash/Bielik-1.5B-v3.0-Instruct-GGUF/tree/main) and place in Summary directory.

3. Open `config.py`, which contains the project's configuration, global variables, and settings. Here you can:
- change the LLM model,
- specify input file names,
- adjust the temperature parameter.
4. Run notebook.
---

Import packages

In [22]:
import os
import csv
import time
from llama_cpp import Llama

Import configuration and set model path

In [33]:
from config import (
    MODEL_CTX,
    GPU_LAYERS,
    INPUT_PATH,
    OUTPUT_FILE,
    INPUT_DIR,
    INPUT_FILENAME,
    MAX_TOKENS_CHUNK,
    MAX_TOKENS_FINAL,
    TEMPERATURE,
    CHUNK_SIZE,
    CHUNK_OVERLAP
)

MODEL_PATH = r'Bielik-1.5B-v3.0-Instruct.Q8_0.gguf'
print(MODEL_PATH)

Bielik-1.5B-v3.0-Instruct.Q8_0.gguf


Model Initialization

In [34]:
def load_model():
    """LLM model loading"""
    if not os.path.exists(MODEL_PATH):
        print(MODEL_PATH)
        raise FileNotFoundError(f"Model not found: {MODEL_PATH}")

    print(f"[INFO] Model loading : {os.path.basename(MODEL_PATH)}")
    start = time.time()

    llm = Llama(
        model_path=MODEL_PATH,
        n_ctx=MODEL_CTX,
        n_gpu_layers=GPU_LAYERS,
        verbose=False,
    )

    elapsed = time.time() - start
    print(f"[INFO] Model loaded: {elapsed:.1f}s")
    return llm

Prompt preparing (ChatML — Bielik format)

In [35]:
def create_prompt(system: str, user: str) -> str:
    return (
        f"<|im_start|>system\n{system}<|im_end|>\n"
        f"<|im_start|>user\n{user}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )


Split text to chunks.

It splits the text into overlapping segments.
It attempts to split at sentence boundaries (dot/semicolon).

In [36]:
def chunk_text(text: str, chunk_size: int = CHUNK_SIZE,
               overlap: int = CHUNK_OVERLAP) -> list[str]:
    text = text.strip()
    if not text:
        return []

    # short text - one chunk
    if len(text) <= chunk_size:
        return [text]

    chunks = []
    start = 0

    while start < len(text):
        end = min(start + chunk_size, len(text))

        # bounduary finding
        if end < len(text):
            search_start = max(start, end - chunk_size // 5)
            last_period = text.rfind('.', search_start, end)
            last_semicolon = text.rfind(';', search_start, end)
            best_break = max(last_period, last_semicolon)

            if best_break > start:
                end = best_break + 1  # +1 for dot

        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)

        # overlap
        if end >= len(text):
            break
        start = end - overlap

    return chunks

LLM request

In [37]:
def get_llm_response(llm, prompt: str, max_tokens: int = 256,
                     temperature: float = 0.1) -> str:
    try:
        output = llm(
            prompt,
            max_tokens=max_tokens,
            stop=["<|im_end|>"],
            temperature=temperature,
            repeat_penalty=1.1,  # decraste 
        )
        return output['choices'][0]['text'].strip()
    except Exception as e:
        print(f"    [WARN] Error LLM: {e}")
        return "gen_error"

Prompts in polish

In [38]:
SYSTEM_EXTRACT = "Wyodrębnij branżę, produkty i usługi z opisu firmy."

USER_EXTRACT_TEMPLATE = (
    "Wypisz dane z tekstu w formacie:\n"
    "Branża: [nazwa]\n"
    "Produkty: [max 5, lub 'brak']\n"
    "Usługi: [max 5, lub 'brak']\n\n"
    "Pomijaj adresy i lokalizacje.\n"
    "Jeśli tekst pusty lub 'BRAK OPISU' napisz tylko: BRAK OPISU\n\n"
    "Tekst:\n{text}"
)

SYSTEM_MERGE = "Scal dane w jeden wpis. Usuń duplikaty."

USER_MERGE_TEMPLATE = (
    "Scal w format: Branża: ... | Produkty: max 5 | Usługi: max 5\n"
    "Usuń powtórzenia.\n\n"
    "Dane:\n{fragments}"
)


Description analyze

In [39]:
def analyze_single_text(llm, text: str) -> str:
    text = text.strip()

    # check
    if not text:
        return "NO CONTENT"

    if text.upper() in ("BRAK OPISU", "BRAK", "-", ""):
        return "NO DESCRIPTION"

    # split for chunks
    chunks = chunk_text(text)

    if not chunks:
        return "NO CONTENT"

    # ── STEP 1: chunk analyze (MAP) ──
    partial_results = []

    for idx, chunk in enumerate(chunks, 1):
        if len(chunks) > 1:
            print(f"    Chunk {idx}/{len(chunks)}...")

        user_prompt = USER_EXTRACT_TEMPLATE.format(text=chunk)
        prompt = create_prompt(SYSTEM_EXTRACT, user_prompt)
        result = get_llm_response(
            llm, prompt,
            max_tokens=MAX_TOKENS_CHUNK,
            temperature=TEMPERATURE
        )

        if result and result != "gen_error":
            partial_results.append(result)

    if not partial_results:
        return "NO RESULTS"

    # ── STEP 2: merging (REDUCE) ──
    if len(partial_results) == 1:
        final_result = partial_results[0]
    else:
        combined = "\n---\n".join(partial_results)
        # length limit
        combined = combined[:2500]

        user_prompt = USER_MERGE_TEMPLATE.format(fragments=combined)
        prompt = create_prompt(SYSTEM_MERGE, user_prompt)

        final_result = get_llm_response(
            llm, prompt,
            max_tokens=MAX_TOKENS_FINAL,
            temperature=0.2
        )

    # ── result cleaning for CSV ──
    clean = final_result.replace('\n', ' | ').replace('\r', '')
    clean = clean.replace(';', ',') 
    clean = ' '.join(clean.split())  # delete double space

    return clean

File processing

In [40]:
def count_lines(filepath: str) -> int:
    with open(filepath, 'r', encoding='utf-8') as f:
        return sum(1 for _ in f)


def count_processed_lines(output_path: str) -> int:
    if not os.path.exists(output_path):
        return 0
    with open(output_path, 'r', encoding='utf-8-sig') as f:
        return sum(1 for line in f if line.strip())


def process_file():
    # ── Validation ──
    if not os.path.exists(INPUT_PATH):
        print(f"[Error] Input file doesn't exist: {INPUT_PATH}")
        if not os.path.exists(INPUT_DIR):
            os.makedirs(INPUT_DIR)
            print(f"[INFO] Create dir '{INPUT_DIR}'. "
                  f"Put file in '{INPUT_FILENAME}'.")
        return

    # ── model loading ──
    llm = load_model()

    # ── line counting ──
    total_lines = count_lines(INPUT_PATH)
    already_done = count_processed_lines(OUTPUT_FILE)

    print(f"[INFO] Input file: {INPUT_PATH} ({total_lines} linii)")
    print(f"[INFO] Output file:  {OUTPUT_FILE}")

    if already_done > 0:
        print(f"[INFO] Start again from {already_done + 1} "
              f"(processed: {already_done})")

    # ── processing ──
    processed = 0
    errors = 0
    start_time = time.time()

    # adding ('a') for start again
    write_mode = 'a' if already_done > 0 else 'w'

    with open(INPUT_PATH, 'r', encoding='utf-8') as f_in, \
         open(OUTPUT_FILE, write_mode, encoding='utf-8-sig') as f_out:

        # header for new file
        if write_mode == 'w':
            f_out.write("DANE_ORYGINALNE;WYNIK_AI\n")
            f_out.flush()

        for i, line in enumerate(f_in, 1):
            line = line.strip()

            # skip empty line
            if not line:
                continue

            # skip processed line
            if i <= already_done:
                continue

            # ── progress bar ──
            elapsed = time.time() - start_time
            if processed > 0:
                avg_time = elapsed / processed
                remaining = avg_time * (total_lines - i)
                eta = f"~{remaining/60:.0f}min"
            else:
                eta = "calulating..."

            print(f"\n{'='*60}")
            print(f"[{i}/{total_lines}] Processing... (ETA: {eta})")
            print(f"  Text: {line[:80]}{'...' if len(line)>80 else ''}")

            try:
                # LLM analyze
                line_start = time.time()
                ai_result = analyze_single_text(llm, line)
                line_time = time.time() - line_start

                # Save
                output_line = f"{line};{ai_result}\n"
                f_out.write(output_line)
                f_out.flush()

                processed += 1
                print(f"  [OK] Time: {line_time:.1f}s")
                print(f"  Result: {ai_result[:100]}{'...' if len(ai_result)>100 else ''}")

            except KeyboardInterrupt:
                print(f"\n[STOP] Interrupted by the user after {processed} lines.")
                print(f"[INFO] You can resume — the script will skip the lines that have already been processed.")
                return

            except Exception as e:
                errors += 1
                print(f"  [Error] {type(e).__name__}: {e}")
                f_out.write(f"{line};Error: {str(e)[:50]}\n")
                f_out.flush()

    # ── Summary ──
    total_time = time.time() - start_time
    print(f"\n{'='*60}")
    print(f"[END] Processing done!")
    print(f"  Processed:  {processed} lines")
    print(f"  Errors:         {errors}")
    print(f"  Time:   {total_time/60:.1f} min")
    if processed > 0:
        print(f"  Avarage time:   {total_time/processed:.1f}s / line")
    print(f"  Result in:       {OUTPUT_FILE}")

Start processing

In [ ]:
if __name__ == "__main__":
    process_file()